# Exploring Embeddings with Keras

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
import sys

sys.executable

'/home/thomas/Python/virtualenvs/uj_embeddings/bin/python'

## Imports

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances

In [4]:
import keras
from keras import layers
from keras import ops

import math
import numpy as np
import pandas as pd
#from tensorflow import data as tf_data
#import matplotlib.pyplot as plt
from functools import partial

2024-07-01 08:34:41.657318: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-01 08:34:43.181253: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-01 08:34:46.883673: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-01 08:34:58.213575: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
CSV_HEADER = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "gender",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income_bracket",
]

train_data_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
)
train_data = pd.read_csv(train_data_url, header=None, names=CSV_HEADER)

test_data_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
)
test_data = pd.read_csv(test_data_url, header=None, names=CSV_HEADER)

print(f"Train dataset shape: {train_data.shape}")
print(f"Test dataset shape: {test_data.shape}")

In [ ]:
test_data = test_data[1:]
test_data.income_bracket = test_data.income_bracket.apply(
    lambda value: value.replace(".", "")
)

In [ ]:
train_data_file = "train_data.csv"
test_data_file = "test_data.csv"

train_data.to_csv(train_data_file, index=False, header=False)
test_data.to_csv(test_data_file, index=False, header=False)


In [ ]:
# A list of the numerical feature names.
NUMERIC_FEATURE_NAMES = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
]
# A dictionary of the categorical features and their vocabulary.
CATEGORICAL_FEATURES_WITH_VOCABULARY = {
    "workclass": sorted(list(train_data["workclass"].unique())),
    "education": sorted(list(train_data["education"].unique())),
    "marital_status": sorted(list(train_data["marital_status"].unique())),
    "occupation": sorted(list(train_data["occupation"].unique())),
    "relationship": sorted(list(train_data["relationship"].unique())),
    "race": sorted(list(train_data["race"].unique())),
    "gender": sorted(list(train_data["gender"].unique())),
    "native_country": sorted(list(train_data["native_country"].unique())),
}
# Name of the column to be used as instances weight.
WEIGHT_COLUMN_NAME = "fnlwgt"
# A list of the categorical feature names.
CATEGORICAL_FEATURE_NAMES = list(CATEGORICAL_FEATURES_WITH_VOCABULARY.keys())
# A list of all the input features.
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + CATEGORICAL_FEATURE_NAMES
# A list of column default values for each feature.
COLUMN_DEFAULTS = [
    [0.0] if feature_name in NUMERIC_FEATURE_NAMES + [WEIGHT_COLUMN_NAME] else ["NA"]
    for feature_name in CSV_HEADER
]
# The name of the target feature.
TARGET_FEATURE_NAME = "income_bracket"
# A list of the labels of the target features.
TARGET_LABELS = [" <=50K", " >50K"]


In [ ]:
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
DROPOUT_RATE = 0.2
BATCH_SIZE = 265
NUM_EPOCHS = 15

NUM_TRANSFORMER_BLOCKS = 3  # Number of transformer blocks.
NUM_HEADS = 4  # Number of attention heads.
EMBEDDING_DIMS = 16  # Embedding dimensions of the categorical features.
MLP_HIDDEN_UNITS_FACTORS = [
    2,
    1,
]  # MLP hidden layer units, as factors of the number of inputs.
NUM_MLP_BLOCKS = 2  # Number of MLP blocks in the baseline model.


In [ ]:
target_label_lookup = layers.StringLookup(
    vocabulary=TARGET_LABELS, mask_token=None, num_oov_indices=0
)


def prepare_example(features, target):
    target_index = target_label_lookup(target)
    weights = features.pop(WEIGHT_COLUMN_NAME)
    return features, target_index, weights


lookup_dict = {}
for feature_name in CATEGORICAL_FEATURE_NAMES:
    vocabulary = CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name]
    # Create a lookup to convert a string values to an integer indices.
    # Since we are not using a mask token, nor expecting any out of vocabulary
    # (oov) token, we set mask_token to None and num_oov_indices to 0.
    lookup = layers.StringLookup(
        vocabulary=vocabulary, mask_token=None, num_oov_indices=0
    )
    lookup_dict[feature_name] = lookup


def encode_categorical(batch_x, batch_y, weights):
    for feature_name in CATEGORICAL_FEATURE_NAMES:
        batch_x[feature_name] = lookup_dict[feature_name](batch_x[feature_name])

    return batch_x, batch_y, weights


def get_dataset_from_csv(csv_file_path, batch_size=128, shuffle=False):
    dataset = (
        tf_data.experimental.make_csv_dataset(
            csv_file_path,
            batch_size=batch_size,
            column_names=CSV_HEADER,
            column_defaults=COLUMN_DEFAULTS,
            label_name=TARGET_FEATURE_NAME,
            num_epochs=1,
            header=False,
            na_value="?",
            shuffle=shuffle,
        )
        .map(prepare_example, num_parallel_calls=tf_data.AUTOTUNE, deterministic=False)
        .map(encode_categorical)
    )
    return dataset.cache()


In [ ]:
def run_experiment(
    model,
    train_data_file,
    test_data_file,
    num_epochs,
    learning_rate,
    weight_decay,
    batch_size,
):
    optimizer = keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy")],
    )

    train_dataset = get_dataset_from_csv(train_data_file, batch_size, shuffle=True)
    validation_dataset = get_dataset_from_csv(test_data_file, batch_size)

    print("Start training the model...")
    history = model.fit(
        train_dataset, epochs=num_epochs, validation_data=validation_dataset
    )
    print("Model training finished")

    _, accuracy = model.evaluate(validation_dataset, verbose=0)

    print(f"Validation accuracy: {round(accuracy * 100, 2)}%")

    return history

In [ ]:
def create_model_inputs():
    inputs = {}
    for feature_name in FEATURE_NAMES:
        if feature_name in NUMERIC_FEATURE_NAMES:
            inputs[feature_name] = layers.Input(
                name=feature_name, shape=(), dtype="float32"
            )
        else:
            inputs[feature_name] = layers.Input(
                name=feature_name, shape=(), dtype="int32"
            )
    return inputs


In [ ]:
def encode_inputs(inputs, embedding_dims):
    encoded_categorical_feature_list = []
    numerical_feature_list = []

    for feature_name in inputs:
        if feature_name in CATEGORICAL_FEATURE_NAMES:
            vocabulary = CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name]
            # Create a lookup to convert a string values to an integer indices.
            # Since we are not using a mask token, nor expecting any out of vocabulary
            # (oov) token, we set mask_token to None and num_oov_indices to 0.

            # Convert the string input values into integer indices.

            # Create an embedding layer with the specified dimensions.
            embedding = layers.Embedding(
                input_dim=len(vocabulary), output_dim=embedding_dims
            )

            # Convert the index values to embedding representations.
            encoded_categorical_feature = embedding(inputs[feature_name])
            encoded_categorical_feature_list.append(encoded_categorical_feature)

        else:
            # Use the numerical features as-is.
            numerical_feature = ops.expand_dims(inputs[feature_name], -1)
            numerical_feature_list.append(numerical_feature)

    return encoded_categorical_feature_list, numerical_feature_list


In [ ]:
def create_mlp(hidden_units, dropout_rate, activation, normalization_layer, name=None):
    mlp_layers = []
    for units in hidden_units:
        mlp_layers.append(normalization_layer())
        mlp_layers.append(layers.Dense(units, activation=activation))
        mlp_layers.append(layers.Dropout(dropout_rate))

    return keras.Sequential(mlp_layers, name=name)


In [ ]:
def create_tabtransformer_classifier(
    num_transformer_blocks,
    num_heads,
    embedding_dims,
    mlp_hidden_units_factors,
    dropout_rate,
    use_column_embedding=False,
):
    # Create model inputs.
    inputs = create_model_inputs()
    # encode features.
    encoded_categorical_feature_list, numerical_feature_list = encode_inputs(
        inputs, embedding_dims
    )
    # Stack categorical feature embeddings for the Tansformer.
    encoded_categorical_features = ops.stack(encoded_categorical_feature_list, axis=1)
    # Concatenate numerical features.
    numerical_features = layers.concatenate(numerical_feature_list)

    # Add column embedding to categorical feature embeddings.
    if use_column_embedding:
        num_columns = encoded_categorical_features.shape[1]
        column_embedding = layers.Embedding(
            input_dim=num_columns, output_dim=embedding_dims
        )
        column_indices = ops.arange(start=0, stop=num_columns, step=1)
        encoded_categorical_features = encoded_categorical_features + column_embedding(
            column_indices
        )

    # Create multiple layers of the Transformer block.
    for block_idx in range(num_transformer_blocks):
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dims,
            dropout=dropout_rate,
            name=f"multihead_attention_{block_idx}",
        )(encoded_categorical_features, encoded_categorical_features)
        # Skip connection 1.
        x = layers.Add(name=f"skip_connection1_{block_idx}")(
            [attention_output, encoded_categorical_features]
        )
        # Layer normalization 1.
        x = layers.LayerNormalization(name=f"layer_norm1_{block_idx}", epsilon=1e-6)(x)
        # Feedforward.
        feedforward_output = create_mlp(
            hidden_units=[embedding_dims],
            dropout_rate=dropout_rate,
            activation=keras.activations.gelu,
            normalization_layer=partial(
                layers.LayerNormalization, epsilon=1e-6
            ),  # using partial to provide keyword arguments before initialization
            name=f"feedforward_{block_idx}",
        )(x)
        # Skip connection 2.
        x = layers.Add(name=f"skip_connection2_{block_idx}")([feedforward_output, x])
        # Layer normalization 2.
        encoded_categorical_features = layers.LayerNormalization(
            name=f"layer_norm2_{block_idx}", epsilon=1e-6
        )(x)

    # Flatten the "contextualized" embeddings of the categorical features.
    categorical_features = layers.Flatten()(encoded_categorical_features)
    # Apply layer normalization to the numerical features.
    numerical_features = layers.LayerNormalization(epsilon=1e-6)(numerical_features)
    # Prepare the input for the final MLP block.
    features = layers.concatenate([categorical_features, numerical_features])

    # Compute MLP hidden_units.
    mlp_hidden_units = [
        factor * features.shape[-1] for factor in mlp_hidden_units_factors
    ]
    # Create final MLP.
    features = create_mlp(
        hidden_units=mlp_hidden_units,
        dropout_rate=dropout_rate,
        activation=keras.activations.selu,
        normalization_layer=layers.BatchNormalization,
        name="MLP",
    )(features)

    # Add a sigmoid as a binary classifer.
    outputs = layers.Dense(units=1, activation="sigmoid", name="sigmoid")(features)
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


tabtransformer_model = create_tabtransformer_classifier(
    num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
    num_heads=NUM_HEADS,
    embedding_dims=EMBEDDING_DIMS,
    mlp_hidden_units_factors=MLP_HIDDEN_UNITS_FACTORS,
    dropout_rate=DROPOUT_RATE,
)

print("Total model weights:", tabtransformer_model.count_params())
keras.utils.plot_model(tabtransformer_model, show_shapes=True, rankdir="LR")

In [ ]:
history = run_experiment(
    model=tabtransformer_model,
    train_data_file=train_data_file,
    test_data_file=test_data_file,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
)

# Microcrédito

In [2]:
['cpf9_espec',
 'id_eq3_espec',
 'cnpj_espec',
 'status_espec',
 'ano_mes_dia',
 'qtde_sub',
 'ult_ref_reneg',
 'flag pjampliada', 
 'qtd contratos',
 'qtd cntrt novo',
 'qtd cntrt renov int',
 'qtd_cntrt_renov merc',
 'qtd cntrts_en aberto',
 'qtd_cntrt_reneg',
 'qtd_fpd_reneg',
 'qtd cntrts pivo', 
 'qtd_cntrts_app', 
 'avg_idade_clientes',
 'qtd_cntrts_sld_pos_bc',
 'avg_mob_quit_cntrt', 
 'avg plano_cntrt', 
 'rz_vlr_entregue_c0c1',
 'rz_panis_pa',
 'rz_entregue_pa',
 'rz_entregue_panis',
 'avg_saldo_em_aberto_parc',
 'pct_cntrt_app',
 'pct_cntrts_reneg']

['cpf9_espec',
 'id_eq3_espec',
 'cnpj_espec',
 'status_espec',
 'ano_mes_dia',
 'qtde_sub',
 'ult_ref_reneg',
 'flag pjampliada',
 'qtd contratos',
 'qtd cntrt novo',
 'qtd cntrt renov int',
 'qtd_cntrt_renov merc',
 'qtd cntrts_en aberto',
 'qtd_cntrt_reneg',
 'qtd_fpd_reneg',
 'qtd cntrts pivo',
 'qtd_cntrts_app',
 'avg_idade_clientes',
 'qtd_cntrts_sld_pos_bc',
 'avg_mob_quit_cntrt',
 'avg plano_cntrt',
 'rz_vlr_entregue_c0c1',
 'rz_panis_pa',
 'rz_entregue_pa',
 'rz_entregue_panis',
 'avg saldo_em_aberto_parc',
 'pct_cntrt_app',
 'pct_cntrts_reneg']

- CONSIGNADO
    - Mix de propostas      -> Propostas por GH (2305) (Todos SPI cujas empresas possuem convênio com o banco (nrregra in ('OP','EP')). Propostas APROVADAS = quem possui PA no público acima)
    - Mix de contratação    -> Contratos por GH (2306)
    - Mix de inadimplência  -> Bad por GH
    * No mix de contratação, devemos utilizar o mesmo público de propostas como denominador, ou levamos em conta somente a referência?
    * No mix de contratação, utilizamos somente os contratos internos?
    * No mix de inadimplência, 100% interno? ou 50% 50% com mercado? 
      
- IMOB
    - Mix de propostas      -> Propostas por GH (2303)
    - Mix de contratação    -> Contratos por GH (2304)
    - Mix de inadimplência  -> Bad por GH
 


- PROPOSTAS
  - CONSIG: Cruzar as bases de produto (que identificam o público SPI de empresas conveniadas) com a base do modelo (GH)
  - IMOB: Cruzar base de propostas do produto (SAS, csv) com a base do modelo (GH)
 
- APROVAÇÃO
  - CONSIG: Cruzar a base de propostas acima (já marcada com GH do modelo) com a de produtos (que marcam o PA)
  - IMOB: A base de propostas já traz info de decisao
 
- CONTRATAÇÂO
  - CONSIG: Cruzar a base de aprovação acima (com GH e decisão de proposta) com a base do FOUCAULT CONTRATAÇÃO
  - IMOB: Cruzar a base de propostas acima (com GH e decisão de proposta) com a base do FOUCAULT CONTRATAÇÃO
 
- INADIMPLENCIA
  - CONSIG: Cruzar as bases de contratação acima (com GH, decisão de proposta e info de contratação) com as bases do FOUCAULT PD ITAU e PD MERC
  - IMOB: Cruzar as bases de contratação acima (com GH, decisão de proposta e info de contratação) com as bases do FOUCAULT PD ITAU e PD MERC

In [ ]:
politica_micro = ['Novos','Renov_Interna','Renov_Mercado']

- Acrescentar ao processo, join com a tabela de contratos
  - [ ] Cruzamento com modelo eleito PF (Polaris/VPRO)
  - [ ] Cruzamento com Meridian Microcredito (Capivara)
 
- GRUPO POLARIS
  - VP1: A1 - C3
  - VP2: C4 - D1
  - VP3: D2 - D4
  - VP4: E1 - E2
  - VP5: E3 - E4
  - VP6: F1
  - VP7: F2
  - VP8: F3 - f4

In [5]:
grupo_polaris = {
    'VP1': ['A1','A2','A3','B1','B2','B3','B4','C1','C2','C3'],
    'VP2': ['C4','D1'],
    'VP3': ['D2','D3','D4'],
    'VP4': ['E1','E2'],
    'VP5': ['E3','E4'],
    'VP6': ['F1'],
    'VP7': ['F2'],
    'VP8': ['F3','F4']
}

grupo_polaris
# conditions = [full_table.rating_polaris.isin(v) for k,v in grupo_polaris.items()]
# choices = [k for k,_ in grupo_polaris.items()]

# full_table['grupo_polaris'] == np.select(conditions, choices)

{'VP1': ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'B4', 'C1', 'C2', 'C3'],
 'VP2': ['C4', 'D1'],
 'VP3': ['D2', 'D3', 'D4'],
 'VP4': ['E1', 'E2'],
 'VP5': ['E3', 'E4'],
 'VP6': ['F1'],
 'VP7': ['F2'],
 'VP8': ['F3', 'F4']}

In [16]:
grupo_polaris_hexagono = np.array([
    ['VPH1']*5,
    ['VPH1']*3 + ['VPH2']*2,
    ['VPH1']*2 + ['VPH2']*1 + ['VPH3']*2,
    ['VPH2']*2 + ['VPH3']*1 + ['VPH4']*2,
    ['VPH3']*2 + ['VPH4']*2 + ['VPH5']*1,
    ['VPH4']*2 + ['VPH5']*3,
    ['VPH5']*3 + ['VPH7']*2,
    ['VPH6']*2 + ['VPH7']*1 + ['VPH8']*2
])

df_vph = pd.DataFrame(grupo_polaris_hexagono, index=list(grupo_polaris.keys()), columns=[x for x in range(1,6)])
df_vph

,1,2,3,4,5
VP1,VPH1,VPH1,VPH1,VPH1,VPH1
VP2,VPH1,VPH1,VPH1,VPH2,VPH2
VP3,VPH1,VPH1,VPH2,VPH3,VPH3
VP4,VPH2,VPH2,VPH3,VPH4,VPH4
VP5,VPH3,VPH3,VPH4,VPH4,VPH5
VP6,VPH4,VPH4,VPH5,VPH5,VPH5
VP7,VPH5,VPH5,VPH5,VPH7,VPH7
VP8,VPH6,VPH6,VPH7,VPH8,VPH8


In [12]:
df_vph.at['VP8',1]

'VPH6'

In [31]:
grupo_risco_meridian = np.array([
    ['GRM1']*4 + ['GRM2']*2 + ['GRM4'] + ['GRM5']*2,
    ['GRM1']*1 + ['GRM2']*3 + ['GRM3']*2 + ['GRM4']*1 + ['GRM5']*2,
    ['GRM2']*3 + ['GRM3']*2 + ['GRM4']*2 + ['GRM5']*2,
    ['GRM3']*4 + ['GRM4']*3 + ['GRM5']*2,
    ['GRM4']*5 + ['GRM5']*4,
    ['GRM5']*5 + ['GRM6']*4,
    ['GRM6']*7 + ['GRM7']*2,
    ['GRM6']*5 + ['GRM7']*4
    
    
])

df_grm = pd.DataFrame(grupo_risco_meridian, index=np.sort(pd.unique(df_vph.values.ravel())), columns=[x for x in range(1,10)])
df_grm

,1,2,3,4,5,6,7,8,9
VPH1,GRM1,GRM1,GRM1,GRM1,GRM2,GRM2,GRM4,GRM5,GRM5
VPH2,GRM1,GRM2,GRM2,GRM2,GRM3,GRM3,GRM4,GRM5,GRM5
VPH3,GRM2,GRM2,GRM2,GRM3,GRM3,GRM4,GRM4,GRM5,GRM5
VPH4,GRM3,GRM3,GRM3,GRM3,GRM4,GRM4,GRM4,GRM5,GRM5
VPH5,GRM4,GRM4,GRM4,GRM4,GRM4,GRM5,GRM5,GRM5,GRM5
VPH6,GRM5,GRM5,GRM5,GRM5,GRM5,GRM6,GRM6,GRM6,GRM6
VPH7,GRM6,GRM6,GRM6,GRM6,GRM6,GRM6,GRM6,GRM7,GRM7
VPH8,GRM6,GRM6,GRM6,GRM6,GRM6,GRM7,GRM7,GRM7,GRM7


In [32]:
grupo_risco_renovacao = np.array([
    ['GRR1']*2 + ['GRR2']*7,
    ['GRR1']*1 + ['GRR2']*1 + ['GRR3']*4 + ['GRR4']*3,
    ['GRR2']*2 + ['GRR3']*3 + ['GRR4']*4,
    ['GRR3']*2 + ['GRR4']*3 + ['GRR5']*1 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*4 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*4 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*4 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*2 + ['GRR6']*5    
])

df_grr = pd.DataFrame(grupo_risco_renovacao, index=list(grupo_polaris.keys()), columns=[x for x in range(1,10)])
df_grr

,1,2,3,4,5,6,7,8,9
VP1,GRR1,GRR1,GRR2,GRR2,GRR2,GRR2,GRR2,GRR2,GRR2
VP2,GRR1,GRR2,GRR3,GRR3,GRR3,GRR3,GRR4,GRR4,GRR4
VP3,GRR2,GRR2,GRR3,GRR3,GRR3,GRR4,GRR4,GRR4,GRR4
VP4,GRR3,GRR3,GRR4,GRR4,GRR4,GRR5,GRR6,GRR6,GRR6
VP5,GRR4,GRR4,GRR5,GRR5,GRR5,GRR5,GRR6,GRR6,GRR6
VP6,GRR4,GRR4,GRR5,GRR5,GRR5,GRR5,GRR6,GRR6,GRR6
VP7,GRR4,GRR4,GRR5,GRR5,GRR5,GRR5,GRR6,GRR6,GRR6
VP8,GRR4,GRR4,GRR5,GRR5,GRR6,GRR6,GRR6,GRR6,GRR6


MATRIZES DE APROVAÇAO

In [62]:
class_espec = ['aam','aaa','bbb','ccc','ddd','hhh','ppp']

NOVOS

In [63]:
n_class_espec = len(class_espec)
n_grps_grm = len(pd.unique(df_grm.values.ravel()))

mat_aprov_novos = pd.DataFrame(np.ones((n_grps_grm,n_class_espec)), index=np.sort(pd.unique(df_grm.values.ravel())), columns=class_espec)
novos_neg = [('GRM3','ddd'), 
             ('GRM4','ccc'), ('GRM4','ddd'), 
             ('GRM5','bbb'), ('GRM5','ccc'), ('GRM5','ddd'),
             ('GRM6','aaa'), ('GRM6','bbb'), ('GRM6','ccc'), ('GRM6','ddd'), ('GRM6','hhh'), ('GRM6','ppp'),
             ('GRM7','aam'), ('GRM7','aaa'), ('GRM7','bbb'), ('GRM7','ccc'), ('GRM7','ddd'), ('GRM7','hhh'), ('GRM7','ppp')]

for grm, clasf in novos_neg:
    mat_aprov_novos.loc[grm,clasf] = 0

mat_aprov_novos

,aam,aaa,bbb,ccc,ddd,hhh,ppp
GRM1,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GRM2,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GRM3,1.0,1.0,1.0,1.0,0.0,1.0,1.0
GRM4,1.0,1.0,1.0,0.0,0.0,1.0,1.0
GRM5,1.0,1.0,0.0,0.0,0.0,1.0,1.0
GRM6,1.0,0.0,0.0,0.0,0.0,0.0,0.0
GRM7,0.0,0.0,0.0,0.0,0.0,0.0,0.0


RENOVACAO INTERNA

In [64]:
pd.unique(df_grr.values.ravel())

array(['GRR1', 'GRR2', 'GRR3', 'GRR4', 'GRR5', 'GRR6'], dtype=object)

In [65]:
n_class_espec, n_grps_grr

(7, 6)

In [66]:
len(class_espec)

7

In [67]:
n_class_espec = len(class_espec)
n_grps_grr = len(pd.unique(df_grr.values.ravel()))

mat_aprov_renov_int = pd.DataFrame(np.ones((n_grps_grr,n_class_espec)), 
                                   index=np.sort(pd.unique(df_grr.values.ravel())), 
                                   columns=class_espec)
renov_int_neg = [('GRR6','ccc'), ('GRR6','ddd')]

for grr, clasf in renov_int_neg:
    mat_aprov_renov_int.loc[grr,clasf] = 0

mat_aprov_renov_int.astype(bool)

,aam,aaa,bbb,ccc,ddd,hhh,ppp
GRR1,True,True,True,True,True,True,True
GRR2,True,True,True,True,True,True,True
GRR3,True,True,True,True,True,True,True
GRR4,True,True,True,True,True,True,True
GRR5,True,True,True,True,True,True,True
GRR6,True,True,True,False,False,True,True


In [55]:
np.ones((n_grps_grr,n_class_espec))

array([[1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.]])

In [ ]:
full_table.politica_micro

In [ ]:
conditions = [arr1 > 2, arr2 < 9]
choices = [arr1, arr2 * 2]

result = np.select(conditions, choices, default=0)

- Propostas
  - Hexagono
  - Rating Polaris
  - Rating Meridian
  - Classificação do espec
  - Bad
 
- Métricas
  - Propostas aceitas/negadas
  - Propostas aceitas: calcular bad
  - Propostas negadas: verificar se houve contratação no mercado e, caso sim, marcar bad
 

- Determinar ponto de corte do score que mantém a média da bad no geral

- Ao determinar os cut points dos grupos de classificação, manter a mesma proporção de propostas aceitas/negadas? Ou manter a inadimplência (indicadores NPL e FPD) de cada classificação nos mesmos níveis?

In [ ]:
- Marca propostas aprovadas
    - Calcula bad
- Marca propostas negadas
    - Verifica contratação no mercado
    - Calcula bad mercado

- Marca a nova classificação dos especs
- Marca propostas que teriam sido aprovadas
    - Calcula bad
    - Verifica bad das propostas SWAP IN
- Marca propostas que teriam sido negadas
    - Verifica bad das propostas SWAP OUT (total e out)

- Tirar a duplicidade da base de propostas
  - Cruzar a base de contratos (cpf cliente, safra, valor) (LEFT JOIN) com a de propostas, pegando ID de transacao
  - Exclui da base inicial de pŕopostas todos os IDs transação já verificados na base de contratos
  - Da base completa de contratos: filtra cpf e safra sem duplicidade
  - Exclui da base inicial de propostas os casos acima
  - Na base de propostas restante: priorizar (cpf, safra) de quem possui proposta aceita

In [ ]:
espec_mcrcrdt_contratos.columns

In [ ]:
espec_mcrcrdt_contratos['cpf_cliente'] = espec_mcrcrdt_contratos.cpf_cliente
espec_mcrcrdt_contratos['cpf_espec'] = espec_mcrcrdt_contratos.cpf_espec
espec_mcrcrdt_contratos['classificacao_espec'] = espec_mcrcrdt_contratos.classificacao_espec
espec_mcrcrdt_contratos['safra'] = espec_mcrcrdt_contratos.safra
espec_mcrcrdt_contratos['valor'] = espec_mcrcrdt_contratos.valor

In [ ]:
espec_mcrcrdt_propostas.columns

In [ ]:
espec_mcrcrdt_propostas['cpf_cliente'] = espec_mcrcrdt_propostas.cpf_cliente
espec_mcrcrdt_propostas['cpf_espec'] = espec_mcrcrdt_propostas.cpf_espec
espec_mcrcrdt_propostas['classificacao_espec'] = espec_mcrcrdt_propostas.classificacao_espec
espec_mcrcrdt_propostas['safra'] = espec_mcrcrdt_propostas.safra
espec_mcrcrdt_propostas['valor'] = espec_mcrcrdt_propostas.valor

### Identifica contratos na base de proposta

In [ ]:
join_cols = ['cpf_cliente', 'cpf_espec', 'classificacao_espec','safra', 'valor']

In [ ]:
join_prop_cntrt = espec_mcrcrdt_contratos[join_cols].merge(espec_mcrcrdt_propostas[join_cols + ['id_transacao']],
                                                          how='inner',
                                                          on=join_cols)

In [ ]:
filtro_prop1 = list(join_prop_cntrt.id_transacao)

### Identifica CPFs com contratos ausentes da base de proposta

In [ ]:
pk = ['cpf_cliente', 'safra'] 

In [ ]:
clientes_contratos = espec_mcrcrdt_contratos[pk].drop_duplicates()

In [ ]:
filtro_prop2 = list(espec_mcrcrdt_propostas[espec_mcrcrdt_propostas.set_index(pk).index.isin(clientes_contratos.set_index(pk).index)].id_transacao)

### Retira as propostas acima da base

In [ ]:
filtro_props = list(set(filtro_prop1+filtro_prop2))

In [ ]:
espec_mcrcrdt_prop_filtr = espec_mcrcrdt_propostas[~(espec_mcrcrdt_propostas.id_transacao.isin(filtro_props))]

### Prioriza as prop restantes por prop aceita

In [ ]:
espec_mcrcrdt_prop_ddup = espec_mcrcrdt_prop_filtr.sort_values(by=['cpf_cliente', 'safra','decisao_final'],na_position='last', ascending=[True,True,True])./
                                        drop_duplicates(subset=['cpf_cliente','safra'])

In [ ]:
espec_mcrcrdt_prop_ddup.shape, espec_mcrcrdt_prop_ddup[['cpf_cliente', 'safra']].drop_duplicates().shape 

In [ ]:
filtro_prop3 = list(espec_mcrcrdt_prop_ddup.id_transacao)

In [ ]:
ids_transac_filtr = list(set(filtro_prop1 + filro_prop3))

In [ ]:
base_final_propostas = espec_mcrcrdt_propostas[espec_mcrcrdt_propostas.id_transacao.isin(ids_transac_filtr)]

In [ ]:
conditions = [full_table.rating_polaris.isin(v) for k,v in grupo_polaris.items()]
choices = [k for k,_ in grupo_polaris.items()]

full_table['grupo_polaris'] == np.select(conditions, choices)


cond_pol_ctrt_novo = base_final_propostas.politica_micro == 'Novos'
cond_pol_ctrt_renov_int = base_final_propostas.politica_micro == 'Renovacao_Interna'clclaro_naoaro_nao
cond_pol_ctrt_renov_mer = base_final_propostas.politica_micro == 'Renovacao_Mercado'

cond_grm7 = base_final_propostas.grm == 7
cond_grm6 = base_final_propostas.grm == 6
cond_grm5 = base_final_propostas.grm == 5
cond_grm4 = base_final_propostas.grm == 4
cond_grm3 = base_final_propostas.grm == 3

cond_grr6 = base_final_propostas.grr == 6
cond_grr5 = base_final_propostas.grr == 5
cond_grr4 = base_final_propostas.grr == 4

cond_class_espec_am = base_final_propostas.classif_espec == 'aam'
cond_class_espec_a = base_final_propostas.classif_espec == 'aaa'
cond_class_espec_b = base_final_propostas.classif_espec == 'bbb'
cond_class_espec_c = base_final_propostas.classif_espec == 'ccc'
cond_class_espec_d = base_final_propostas.classif_espec == 'ddd'
cond_class_espec_h = base_final_propostas.classif_espec == 'hhh'

cond_rdg_gt_30 = base_final_propostas.rdg > 30



claro_nao_novos = (cond_pol_ctrt_novo) & /
                   ((cond_grm7) | /
                    ((cond_grm6) & ~(cond_class_espec_am)) | /
                    ((cond_grm5) & ((cond_class_espec_b) | (cond_class_espec_c) | (cond_class_espec_d))) | /
                    ((cond_grm4) & ((cond_class_espec_c) | (cond_class_espec_d))) | /
                    ((cond_grm3) & (cond_class_espec_d))
                   )

claro_nao_renov_int = (cond_pol_ctrt_renov_int) & /
                       (
                         ((cond_grr6) & ((cond_class_espec_c) | (cond_class_espec_d))) | /
                           ((cond_rdg_gt_30) & /
                            (
                             ((cond_grr6) & (cond_class_espec_b)) | /
                             ((cond_grr5) & ((cond_class_espec_c) | (cond_class_espec_d))) | /
                             ((cond_grr4) & (cond_class_espec_d))
                            )
                           )
                       )


claro_nao_renov_merc = (cond_pol_ctrt_renov_mer) & /
                       (
                        ((cond_grm7) & ~((cond_class_espec_am) | (cond_class_espec_a))) | /
                        ((cond_grm6) & ((cond_class_espec_c) | (cond_class_espec_d) | (cond_class_espec_h))) | /
                        ((cond_grm5) & ((cond_class_espec_c) | (cond_class_espec_d))) | /
                        ((cond_grm4) & (cond_class_espec_d))
                       )   

claro_nao = (claro_nao_novos) | (claro_nao_renov_int) | (claro_nao_renov_mer)



base_final_propostas['prop_aprov'] = np.where(base_final_propostas[claro_nao], 0, 1)



# CLARO NÃO
sting Algorithms to Mitigate ...
￼
Towards Data Science
https://towardsdatascience.com › ...
Nov 8, 2021 — It is easy to create the learning c
# - Novos
#   - GRM7
#   - GRM6 (exceto espec aam)
#   - GRM5 (bbb, ccc, ddd)
#   - GRM4 (ccc, ddd)
#   - GRM3 (ddd)
 
# - Renovação Interna
#   - Todos: GRR6 (ccc, ddd)
#   - rdg 30 a 99
#     - GRR6 (bbb)
#     - GRR5 (ccc,ddd)
#     - GRR4 (ddd)

# - Renovação Mercado
#   - GRM7 (exceto aam e aaa)
#   - GRM6 (ccc, ddd, hhh)
#   - GRM5 (ccc, ddd)
#   - GRM4 (ddd)

In [ ]:
base_final_propostas['modelo_score'] = lgbm.predict_proba()[:1]

In [ ]:
base_final_propostas['nova_marcacao_claf_espec'] = #DECIDIR THRESHOLDS/CUT POINTS

In [ ]:
base_final_propostas['nova_prop_aprov'] = np.where(base_final_propostas[claro_nao], 0, 1)

- MICROCRÉDITO
  - [X] Revisar o target (ever30em3 - se tornou ddd em algum momento nos próximos 3 meses) 
  - [X] Variáveis PJ - Sinal Vital PJ
  - [ ] Oportunidades de modelagem
      - [ ] Reamostrar os datasets de treino e valid levando em conta dimensão temporal
      - [ ] Fazer validação cruzada utilizando TimeSeriesSplit
      - [ ] Ativar earlyStopping no treino
      - [ ] Plotar as curvas de aprendizagem de treino e validação ao longo do tempo (erro X iterações do algoritmo)
      - [ ] Plotar as curvas de aprendizagem de treino e validação ao longo da experiência (erro X tamanho do conjunto de treino)
  - [ ] Avaliar gini somente no público hhh
  - [ ] Construir arvores pequenas somente com o público hhh
  - [ ] Gerar visão de threshold (Karina - se definir escore de corte em X vou colocar Y% a mais/menos o que traz um resultado de Z% a mais/menos)
  - [ ] Explicabilidade para os diferentes grupos (todos, vs hhh)
 
  
  - [ ] Construir matrizes de Grupos de Risco de acordo com a política
    - [ ] Caclular inadimplência (gini) com a classificação deles VS com a nova classificação proposta

- Pegar o último modelo e escorar todos os especs
- Montar GHs
- Calcular a bad de cada classificaçao por safra usando class e marcacao bad deles
  - Comparar com a bad de cada uma das nossas classificações propostas
- Média do nosso score para cada rating deles

- Avaliar método de cálculo de feature importance para Randdom Forest apresentado no livro statistics for data scientists
- Reamostragem dos conjuntos de teste e validação: deixar as últimas 4/5 refs para validação e teste e gerar o conjunto de validação a partir de uma amostra estratificada do conjunto de teste
- Learning curves: para modelos mais simples, variar o tamanho do conjunto de treino e medir o erro no conjunto de treino e teste. Já para os modelos de boosting (ou iterativos, no geral), basta variar a quantidade de iterações do modelo
- Para redução da variância: se o custo computacional não é um problema, vale mais a pena focar os esforços em acrescentar/otimizar parâmetros de regularização do modelo do que em tentar usar modelos mais simples
- Nos algs de boosting: brincar com learning rate e subsample
- Random Fores: parâmetros oob_score e bootstrap

In [ ]:
Random Forest
- n_estimators
- max_depth
- min_samples_split
- min_samples_leaf
- max features
- 120, 300, 500, 800, 1200
- 5, 8, 15, 25, 30, None
- 1, 2, 5, 10, 15, 100
- 1, 2, 5, 10
- log2, sqrt, None
XGBoost
- eta
- gamma
- max_depth
- min_child_weight
- subsample
- colsample_bytree
- lambda
- alpha
- 0.01,0.015, 0.025, 0.05, 0.1
- 0.05-0.1,0.3,0.5,0.7,0.9,1.0
- 3, 5, 7, 9, 12, 15, 17, 25
- 1, 3, 5, 7
- 0.6, 0.7, 0.8, 0.9, 1.0
- 0.6, 0.7, 0.8, 0.9, 1.0
- 0.01-0.1, 1.0 , RS*
- 0, 0.1, 0.5, 1.0 RS*

# Métodos

In [ ]:
# STRATIFIED KFOLD CROSS VALIDATION

# import pandas and model_selection module of scikit-learn
import pandas as pd
from sklearn import model_selection
if __name__ == "__main__":
# Training data is in a csv file called train.csv
df = pd.read_csv("train.csv")
# we create a new column called kfold and fill it with -1
df["kfold"] = -1
# the next step is to randomize the rows of the data
df = df.sample(frac=1).reset_index(drop=True)
# fetch targets
y = df.target.values
# initiate the kfold class from model_selection module
kf = model_selection.StratifiedKFold(n_splits=5)
# fill the new kfold column
for f, (t_, v_) in enumerate(kf.split(X=df, y=y)):
df.loc[v_, 'kfold'] = f
# save the new csv with kfold column
df.to_csv("train_folds.csv", index=False)


In [ ]:
# FEATURE SELECTION - GET RID OF LOW VARIACE FEATURES

from sklearn.feature_selection import VarianceThreshold
data = ...
var_thresh = VarianceThreshold(threshold=0.1)
transformed_data = var_thresh.fit_transform(data)
# transformed data will have all columns with variance less
# than 0.1 removed


In [ ]:
# ACCESSING CORRELATION

import pandas as pd
from sklearn.datasets import fetch_california_housing
# fetch a regression dataset
data = fetch_california_housing()
X = data["data"]
col_names = data["feature_names"]
y = data["target"]
# convert to pandas dataframe
df = pd.DataFrame(X, columns=col_names)
# introduce a highly correlated column
df.loc[:, "MedInc_Sqrt"] = df.MedInc.apply(np.sqrt)
# get correlation matrix (pearson)
df.corr()


In [ ]:

from sklearn.feature_selection import chi2
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import SelectPercentile

class UnivariateFeatureSelction:
    def __init__(self, n_features, problem_type, scoring):
        """
        Custom univariate feature selection wrapper on
        different univariate feature selection models from
        scikit-learn.
        :param n_features: SelectPercentile if float else SelectKBest
        :param problem_type: classification or regression
        :param scoring: scoring function, string
        """
        # for a given problem type, there are only
        # a few valid scoring methods
        # you can extend this with your own custom
        # methods if you wish
        if problem_type == "classification":
            valid_scoring = {
            "f_classif": f_classif,
            "chi2": chi2,
            "mutual_info_classif": mutual_info_classif
            }
            else:
            valid_scoring = {
            "f_regression": f_regression,
            "mutual_info_regression": mutual_info_regression
            }
        
         # raise exception if we do not have a valid scoring method
         if scoring not in valid_scoring:
         raise Exception("Invalid scoring function")
        
         # if n_features is int, we use selectkbest
         # if n_features is float, we use selectpercentile
         # please note that it is int in both cases in sklearn
         if isinstance(n_features, int):
             self.selection = SelectKBest(
             valid_scoring[scoring],
             k=n_features
        
         elif isinstance(n_features, float):
             self.selection = SelectPercentile(
                 valid_scoring[scoring],
                 percentile=int(n_features * 100)
             )
         else:
             raise Exception("Invalid type of feature")
    
     # same fit function
     def fit(self, X, y):
         return self.selection.fit(X, y)
    
     # same transform function
     def transform(self, X):
         return self.selection.transform(X)
    
     # same fit_transform function
     def fit_transform(self, X, y):
         return self.selection.fit_transform(X, y)


ufs = UnivariateFeatureSelction(
 n_features=0.1,
 problem_type="regression",
 scoring="f_regression"
)
ufs.fit(X, y)
X_transformed = ufs.transform(X)

In [ ]:
rf = RandomForestClassifier(n_estimators=500)
scores = defaultdict(list)

# cross-validate the scores on a number of different random splits of the data
for _ in range(3):
 train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.3)
 rf.fit(train_X, train_y)
 acc = metrics.accuracy_score(valid_y, rf.predict(valid_X))
 for column in X.columns:
     X_t = valid_X.copy()
     X_t[column] = np.random.permutation(X_t[column].values)
     shuff_acc = metrics.accuracy_score(valid_y, rf.predict(X_t))
     scores[column].append((acc-shuff_acc)/acc)

df = pd.DataFrame({
 'feature': X.columns,
 'Accuracy decrease': [np.mean(scores[column]) for column in X.columns],
 'Gini decrease': rf_all.feature_importances_,
})

df = df.sort_values('Accuracy decrease')

fig, axes = plt.subplots(ncols=2, figsize=(8, 4.5))

ax = df.plot(kind='barh', x='feature', y='Accuracy decrease', legend=False, ax=axes[0])
ax.set_ylabel('')
ax = df.plot(kind='barh', x='feature', y='Gini decrease', legend=False, ax=axes[1])
ax.set_ylabel('')
ax.get_yaxis().set_visible(False)

In [ ]:
# greedy.py
class GreedyFeatureSelection:
    """
    A simple and custom class for greedy feature selection.
    You will need to modify it quite a bit to make it suitable
    for your dataset.
    """
    
    def evaluate_score(self, X, y):
        """
        This function evaluates model on data and returns
        Area Under ROC Curve (AUC)
        NOTE: We fit the data and calculate AUC on same data.
        WE ARE OVERFITTING HERE.
        But this is also a way to achieve greedy selection.
        k-fold will take k times longer.
        If you want to implement it in really correct way,
        calculate OOF AUC and return mean AUC over k folds.
        This requires only a few lines of change and has been
        shown a few times in this book.
        :param X: training data
        :param y: targets
        :return: overfitted area under the roc curve
        """
        # fit the logistic regression model,
        # and calculate AUC on same data
        # again: BEWARE
        # you can choose any model that suits your data
        model = linear_model.LogisticRegression()
        model.fit(X, y)
        predictions = model.predict_proba(X)[:, 1]
        auc = metrics.roc_auc_score(y, predictions)
        return auc


    def _feature_selection(self, X, y):
        """
        This function does the actual greedy selection
        :param X: data, numpy array
        :param y: targets, numpy array
        :return: (best scores, best features)
        """
        # initialize good features list
        # and best scores to keep track of both
        good_features = []
        best_scores = []

        # calculate the number of features
        num_features = X.shape[1]

        # infinite loop
        while True:
            # initialize best feature and score of this loop
            this_feature = None
            best_score = 0
            # loop over all features
            for feature in range(num_features):
                # if feature is already in good features,
                # skip this for loop
                if feature in good_features:
                    continue
                # selected features are all good features till now
                # and current feature
                selected_features = good_features + [feature]
                # remove all other features from data
                xtrain = X[:, selected_features]
                # calculate the score, in our case, AUC
                score = self.evaluate_score(xtrain, y)
                # if score is greater than the best score
                # of this loop, change best score and best feature
                if score > best_score:
                    this_feature = feature
                    best_score = score
                    # if we have selected a feature, add it
                    # to the good feature list and update best scores list
                if this_feature != None:
                    good_features.append(this_feature)
                    best_scores.append(best_score)
                    # if we didnt improve during the previous round,
                    # exit the while loop
                    if len(best_scores) > 2:
                        if best_scores[-1] < best_scores[-2]:
                            break
                        
            # return best scores and good features
            # why do we remove the last data point?
            return best_scores[:-1], good_features[:-1]


    def __call__(self, X, y):
        """
        Call function will call the class on a set of arguments
        """
        # select features, return scores and selected indices
        scores, features = self._feature_selection(X, y)
        # transform data with selected features
        return X[:, features], scores
    
if __name__ == "__main__":
    # generate binary classification data
    X, y = make_classification(n_samples=1000, n_features=100)
    # transform data by greedy feature selection
    X_transformed, scores = GreedyFeatureSelection()(X, y)


In [ ]:
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.datasets import fetch_california_housing
# fetch a regression dataset
data = fetch_california_housing()
X = data["data"]
col_names = data["feature_names"]
y = data["target"]
# initialize the model
model = LinearRegression()
# initialize RFE
rfe = RFE(
estimator=model,
n_features_to_select=3
)
# fit RFE
rfe.fit(X, y)
# get the transformed data with
# selected columns
X_transformed = rfe.transform(X)

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
# fetch a regression dataset
# in diabetes data we predict diabetes progression
# after one year based on some features
data = load_diabetes()
X = data["data"]
col_names = data["feature_names"]
y = data["target"]
# initialize the model
model = RandomForestRegressor()
# fit the model
model.fit(X, y)

importances = model.feature_importances_
idxs = np.argsort(importances)
plt.title('Feature Importances')
plt.barh(range(len(idxs)), importances[idxs], align='center')
plt.yticks(range(len(idxs)), [col_names[i] for i in idxs])
plt.xlabel('Random Forest Feature Importance')
plt.show()


#### Emsembling Models

In [ ]:
class OptimizeAUC:
    """
    Class for optimizing AUC.
    This class is all you need to find best weights for
    any model and for any metric and for any types of predictions.
    With very small changes, this class can be used for optimization of
    weights in ensemble models of _any_ type of predictions
    """

    def __init__(self):
        self.coef_ = 0
    
    def _auc(self, coef, X, y):
        """
         This functions calulates and returns AUC.
         :param coef: coef list, of the same length as number of models
         :param X: predictions, in this case a 2d array
         :param y: targets, in our case binary 1d array
         """
        # multiply coefficients with every column of the array
        # with predictions.
        # this means: element 1 of coef is multiplied by column 1
        # of the prediction array, element 2 of coef is multiplied
        # by column 2 of the prediction array and so on!
        x_coef = X * coef
        # create predictions by taking row wise sum
        predictions = np.sum(x_coef, axis=1)
    
        # calculate auc score
        auc_score = metrics.roc_auc_score(y, predictions)
        # return negative auc
        return -1.0 * auc_score

    def fit(self, X, y):
        # remember partial from hyperparameter optimization chapter?
        loss_partial = partial(self._auc, X=X, y=y)

        # dirichlet distribution. you can use any distribution you want
        # to initialize the coefficients
        # we want the coefficients to sum to 1
        initial_coef = np.random.dirichlet(np.ones(X.shape[1]), size=1)
        # use scipy fmin to minimize the loss function, in our case auc
        self.coef_ = fmin(loss_partial, initial_coef, disp=True)

    def predict(self, X):
        # this is similar to _auc function
        x_coef = X * self.coef_
        predictions = np.sum(x_coef, axis=1)
        return predictions

In [ ]:
# make a binary classification dataset with 10k samples
# and 25 features
X, y = make_classification(n_samples=10000, n_features=25)
# split into two folds (for this example)
xfold1, xfold2, yfold1, yfold2 = model_selection.train_test_split(
    X,
    y,
    test_size=0.5,
    stratify=y
)
# fit models on fold 1 and make predictions on fold 2
# we have 3 models:
# logistic regression, random forest and xgboost
logreg = linear_model.LogisticRegression()
rf = ensemble.RandomForestClassifier()
xgbc = xgb.XGBClassifier()
# fit all models on fold 1 data
logreg.fit(xfold1, yfold1)
rf.fit(xfold1, yfold1)
xgbc.fit(xfold1, yfold1)
# predict all models on fold 2
# take probability for class 1
pred_logreg = logreg.predict_proba(xfold2)[:, 1]
pred_rf = rf.predict_proba(xfold2)[:, 1]
pred_xgbc = xgbc.predict_proba(xfold2)[:, 1]


# create an average of all predictions
# that is the simplest ensemble
avg_pred = (pred_logreg + pred_rf + pred_xgbc) / 3
# a 2d array of all predictions
fold2_preds = np.column_stack((
    pred_logreg,
    pred_rf,
    pred_xgbc,
    avg_pred
))
# calculate and store individual AUC values
aucs_fold2 = []
for i in range(fold2_preds.shape[1]):
    auc = metrics.roc_auc_score(yfold2, fold2_preds[:, i])
    aucs_fold2.append(auc)
print(f"Fold-2: LR AUC = {aucs_fold2[0]}")
print(f"Fold-2: RF AUC = {aucs_fold2[1]}")
print(f"Fold-2: XGB AUC = {aucs_fold2[2]}")
print(f"Fold-2: Average Pred AUC = {aucs_fold2[3]}")
# now we repeat the same for the other fold
# this is not the ideal way, if you ever have to repeat code,
# create a function!
# fit models on fold 2 and make predictions on fold 1
logreg = linear_model.LogisticRegression()
rf = ensemble.RandomForestClassifier()
xgbc = xgb.XGBClassifier()
logreg.fit(xfold2, yfold2)
rf.fit(xfold2, yfold2)
xgbc.fit(xfold2, yfold2)
pred_logreg = logreg.predict_proba(xfold1)[:, 1]
pred_rf = rf.predict_proba(xfold1)[:, 1]
pred_xgbc = xgbc.predict_proba(xfold1)[:, 1]
avg_pred = (pred_logreg + pred_rf + pred_xgbc) / 3
fold1_preds = np.column_stack((
    pred_logreg,
    pred_rf,
    pred_xgbc,
    avg_pred
))


aucs_fold1 = []
for i in range(fold1_preds.shape[1]):
    auc = metrics.roc_auc_score(yfold1, fold1_preds[:, i])
    aucs_fold1.append(auc)
print(f"Fold-1: LR AUC = {aucs_fold1[0]}")
print(f"Fold-1: RF AUC = {aucs_fold1[1]}")
print(f"Fold-1: XGB AUC = {aucs_fold1[2]}")
print(f"Fold-1: Average prediction AUC = {aucs_fold1[3]}")
# find optimal weights using the optimizer
opt = OptimizeAUC()
# dont forget to remove the average column
opt.fit(fold1_preds[:, :-1], yfold1)
opt_preds_fold2 = opt.predict(fold2_preds[:, :-1])
auc = metrics.roc_auc_score(yfold2, opt_preds_fold2)
print(f"Optimized AUC, Fold 2 = {auc}")
print(f"Coefficients = {opt.coef_}")
opt = OptimizeAUC()
opt.fit(fold2_preds[:, :-1], yfold2)
opt_preds_fold1 = opt.predict(fold1_preds[:, :-1])
auc = metrics.roc_auc_score(yfold1, opt_preds_fold1)
print(f"Optimized AUC, Fold 1 = {auc}")
print(f"Coefficients = {opt.coef_}")


### HP tuning using XGBooster

In [ ]:
idx = np.random.choice(range(5), size=len(X), replace=True)
error = []
for eta, max_depth in product([0.1, 0.5, 0.9], [3, 6, 9]):
    xgb = XGBClassifier(objective='binary:logistic', n_estimators=250, 
                        max_depth=max_depth, learning_rate=eta,
                        eval_metric='error')
    cv_error = []
    for k in range(5):
        fold_idx = idx == k
        train_X = X.loc[~fold_idx]; train_y = y[~fold_idx]
        valid_X = X.loc[fold_idx]; valid_y = y[fold_idx]

        xgb.fit(train_X, train_y)
        pred = xgb.predict_proba(valid_X)[:, 1]
        cv_error.append(np.mean(abs(valid_y - pred) > 0.5))
    error.append({
        'eta': eta,
        'max_depth': max_depth,
        'avg_error': np.mean(cv_error)
    })
    print(error[-1])
errors = pd.DataFrame(error)
print(errors)

### LGBM specific Optuna Pruner

In [ ]:
"""
Optuna example that demonstrates a pruner for LightGBM.

In this example, we optimize the validation accuracy of cancer detection using LightGBM.
We optimize both the choice of booster model and their hyperparameters. Throughout
training of models, a pruner observes intermediate results and stop unpromising trials.

You can run this example as follows:
    $ python lightgbm_integration.py

"""

import numpy as np
import optuna

import lightgbm as lgb
import sklearn.datasets
import sklearn.metrics
from sklearn.model_selection import train_test_split


# FYI: Objective functions can take additional arguments
# (https://optuna.readthedocs.io/en/stable/faq.html#objective-func-additional-args).
def objective(trial):
    data, target = sklearn.datasets.load_breast_cancer(return_X_y=True)
    train_x, valid_x, train_y, valid_y = train_test_split(data, target, test_size=0.25)
    dtrain = lgb.Dataset(train_x, label=train_y)
    dvalid = lgb.Dataset(valid_x, label=valid_y)

    param = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    # Add a callback for pruning.
    pruning_callback = optuna.integration.LightGBMPruningCallback(trial, "auc")
    gbm = lgb.train(param, dtrain, valid_sets=[dvalid], callbacks=[pruning_callback])

    preds = gbm.predict(valid_x)
    pred_labels = np.rint(preds)
    accuracy = sklearn.metrics.accuracy_score(valid_y, pred_labels)
    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10), direction="maximize"
    )
    study.optimize(objective, n_trials=100)

    print("Number of finished trials: {}".format(len(study.trials)))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: {}".format(trial.value))

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

# Auto GH

In [5]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt


def plot_gh(df, ref, perf, gh, perf_denominator=None):
    #Plot GH distribution and performance over time.
    # Initiate subplots
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    # calculate percentage of GH by ref 
    df_count = df.groupby([ref, gh], as_index=False).apply( lambda x: pd.Series({"percentage": len(x)}))
    
    df_count["percentage"] = df_count["percentage"] / df_count.groupby(ref)[
        "percentage"
    ].transform(sum)
    
    # Plots GH distribution over time 
    df_count.pivot_table("percentage", ref, gh).plot( 
        kind="bar", stacked=True, ax=ax[0])
    ax[0].legend(loc="lower left")
    ax[0].set_title("GH distribution over time")
    
    # Calculate mean perf by ref and GH
    if perf_denominator is None:
        df_perf = df.groupby([ref, gh], as_index=False) [perf].mean()
    else:
        df_perf = df.groupby([ref, gh], as_index=False) [perf].sum()
        df_perf[perf] = ( df_perf[perf] /df.groupby([ref, gh]) [perf_denominator].sum().values)
        
    # Plots performance over time
    df_perf[ref] = df_perf[ref].astype(str)
    df_perf.pivot_table(perf, ref, gh).plot(ax=ax[1])
    ax[1].legend(loc="lower left")
    ax[1].set_title("Performance over time")

    # Calculate Log odds by ref and GH
    df_odds = (
        df.groupby([ref, gh]) .apply(lambda x: np.log( x[x[perf] > 0].shape[0] / x[~(x[perf] > 0)].shape[0]
    )).reset_index() .rename(columns={0: "log_odds"}))
    # Plots performance over time
    df_odds[ref] = df_odds[ref].astype(str) 
    df_odds.pivot_table("log_odds", ref, gh).plot(ax=ax[2])
    ax[2].legend(loc="lower left")
    ax[2].set_title("Log odds over time")
    # Rotate x labels
    for axis in ax: axis.tick_params(axis="x", labelrotation=90)
                                                           

In [41]:
def check_gh(
    df,
    ref,
    perf,
    gh,
    gh_min_size=0.05,
    z_min=2,
    perf_denominator=None
):
    # Calculate percentage of GH by ref 
    df_count = df.groupby(by=[ref, gh])[[ref, gh]].apply( lambda x: pd.Series({"percentage": len(x)}))
    df_count["percentage"] = df_count["percentage"] / df_count.groupby(ref)["percentage"].transform('sum')
    # print(df_count)
    df_count.reset_index(inplace=True)
    
    # Calculate mean perf by ref and GH
    if perf_denominator is None: 
        df_perf = df.groupby([ref, gh])[perf].mean()
        df_perf = df_perf.reset_index()
        df_perf.columns = list(df_perf.columns)[:-1] + ['PERF']
    else: 
        df_perf = df.groupby([ref, gh])[perf].mean()
        df_perf = df_perf.reset_index()
        df_perf.columns = list(df_perf.columns)[:-1] + ['PERF']
        df_perf[perf] = (df_perf[perf]/df.groupby([ref, gh])[perf_denominator].sum().values)

    # Generate the check report by GH and calculating 
    report = df_count.groupby(gh, as_index=False)[
        "percentage"
    ].mean() # Mean percentage by GH
    
    # Mean perf by GH
    report["mean_perf"] = df_perf.groupby(gh)[perf].mean().values
    #Std dev of perf by GH
    report["std_perf"] = df_perf.groupby(gh)[perf].std().values
    # Mean perf of previous GH
    report["mean_prev"] = report["mean_perf"].shift(1)
    # Std dev of perf of previous GH
    report["std_prev"] = report["std_perf"].shift(1)
    
    # Calculate z-score
    report["z"] = (report["mean_perf"] - report["mean_prev"]) / ((report["std_perf"] ** 2 + report["std_prev"] ** 2) ** 0.5)
    #Check column
    report["ok"] = (report["percentage"] > gh_min_size) & ((report["mean_prev"].isna()) | (report["z"] > z_min))
    # Set GH as index
    report = report.set_index(gh)
    return report

In [50]:
def get_cut_points(
    df,
    ref,
    perf,
    score,
    gh_min_size=0.05,
    z_min=2,
    step=0.01,
    perf_denominator=None
):
    # Initial cut points 
    cut_points = [0, 1]
    # Flag that indicates to continue 
    not_end = True
    # Instantiate progress bar 
    pbar = tqdm(total=len(np.arange(step, 1, step)), position=0, leave=True)
    
    while not_end:
        # Set flag to stop if no one new point be found 
        not_end = False
        # Loop thru every step 
        for i in np.arange(cut_points[-2] + step, 1, step):
            # Update progress bar
            pbar.update(1)
            # Insert new point in the end 
            new_cut_points = cut_points[:-1] + [i] + cut_points[-1:]
            # Create GH Labels 
            labels = np.arange(1, len(new_cut_points))
            # Create dataframe with GHS
            gh = pd.DataFrame(
                {"gh": pd.cut(df[score], bins=new_cut_points, labels=labels)}
            )
            
            # Check if all GHS are 'ok' with constraints
            if check_gh(
                pd.concat([df, gh], axis=1),
                ref,
                perf,
                "gh",
                gh_min_size,
                z_min,
                perf_denominator,
            )["ok"].all():
                # Update cut points
                cut_points = new_cut_points
                # Set flat to continue looking for new cut points
                not_end = True
                break

    pbar.close()
    return cut_points

In [51]:
def get_gh(df, score, cut_points):
    ghs = pd.cut(
        df[score], 
        bins=cut_points,
        labels=np.arange(1, len(cut_points))
    )
    return ghs

In [52]:
ref = np.array([2306]*10 + [2307]*10 + [2308]*10 + [2309]*10 + [2310]*10 + [2311]*10)
perf = np.array([0]*25 + [1]*1 + [0]*5 + [1]*1 + [0]*5 + [1]*1 + [0]*5 + [1]*1 + [0]*5 + [1]*1 + [0]*5 + [1]*5)
score = np.linspace(0,1,60)

In [53]:
np.random.shuffle(ref)

In [54]:
df = pd.DataFrame({'SAFRA': ref, 'PERF': perf, 'SCORE': score})

In [55]:
df.SAFRA.value_counts()

SAFRA
2308    10
2309    10
2311    10
2310    10
2306    10
2307    10
Name: count, dtype: int64

In [56]:
df.groupby('SAFRA').SCORE.mean()

SAFRA
2306    0.577966
2307    0.627119
2308    0.391525
2309    0.449153
2310    0.606780
2311    0.347458
Name: SCORE, dtype: float64

In [57]:
cp = get_cut_points(df,'SAFRA','PERF','SCORE', step=0.01)

100%|███████████████████████████████████████████| 99/99 [00:01<00:00, 64.32it/s]


In [58]:
cp

[0, 0.9, 1]

In [60]:
df['GH'] = get_gh(df,'SCORE',cp)

In [61]:
df.GH.value_counts()

GH
1    53
2     6
Name: count, dtype: int64